In [1]:
import math
import torch
import torch.nn as nn

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(device)

mps


In [2]:
class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=1000
    ):
        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            *
            (
                -math.log(10000.0)
                / d_model
            )
        )

        pe[:,0::2] = torch.sin(
            position * div_term
        )

        pe[:,1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self,x):

        return (
            x +
            self.pe[:,:x.size(1)]
        )

In [9]:
class CuneiformTransformer(
    nn.Module
):

    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=256,
        nhead=4,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=512,
        dropout=0.1
    ):
        super().__init__()

        self.d_model = d_model

        self.src_embedding = nn.Embedding(
            src_vocab_size,
            d_model
        )

        self.tgt_embedding = nn.Embedding(
            tgt_vocab_size,
            d_model
        )

        self.positional = (
            PositionalEncoding(
                d_model
            )
        )

        self.transformer = (
            nn.Transformer(
                d_model=d_model,
                nhead=nhead,
                num_encoder_layers=num_encoder_layers,
                num_decoder_layers=num_decoder_layers,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                batch_first=True
            )
        )

        self.fc_out = nn.Linear(
            d_model,
            tgt_vocab_size
        )

    def forward(
        self,
        src,
        tgt
    ):

        src = (
            self.src_embedding(src)
            *
            math.sqrt(
                self.d_model
            )
        )

        tgt = (
            self.tgt_embedding(tgt)
            *
            math.sqrt(
                self.d_model
            )
        )

        src = self.positional(src)
        tgt = self.positional(tgt)

        tgt_mask = (
            nn.Transformer
            .generate_square_subsequent_mask(
                tgt.size(1)
            )
            .to(src.device)
        )

        output = (
            self.transformer(
                src,
                tgt,
                tgt_mask=tgt_mask
            )
        )

        return self.fc_out(output)

In [10]:
import pandas as pd

df = pd.read_csv("../data/mtm24_transliterated.csv")

MAX_SRC_LEN = 200
MAX_TGT_LEN = 300

df = df[
    (df["original_cuneiform"].str.len() <= MAX_SRC_LEN)
    &
    (df["transliteration"].str.len() <= MAX_TGT_LEN)
].copy()

src_chars = set()
for text in df["original_cuneiform"]:
    src_chars.update(text)

tgt_chars = set()
for text in df["transliteration"]:
    tgt_chars.update(text)

src_vocab = {
    "<pad>":0,
    "<sos>":1,
    "<eos>":2,
    "<unk>":3
}

for c in sorted(src_chars):
    src_vocab[c] = len(src_vocab)

tgt_vocab = {
    "<pad>":0,
    "<sos>":1,
    "<eos>":2,
    "<unk>":3
}

for c in sorted(tgt_chars):
    tgt_vocab[c] = len(tgt_vocab)

print(len(src_vocab))
print(len(tgt_vocab))

427
71
